# Pre-Match Keyword Feature Extraction

Loads the cached Guardian articles and scans each article body for injury, suspension, and return keywords.
Produces six per-match features that can be merged onto the modelling dataset by `match_api_id`:

| Feature | Description |
|---|---|
| `home_injury_count` | Injury keyword hits in home team articles |
| `away_injury_count` | Injury keyword hits in away team articles |
| `home_suspension_count` | Suspension keyword hits in home team articles |
| `away_suspension_count` | Suspension keyword hits in away team articles |
| `home_return_count` | Return-from-injury keyword hits in home team articles |
| `away_return_count` | Return-from-injury keyword hits in away team articles |

A `has_coverage` flag is also added for matches where no articles were found at all.

**Note:** Articles are attributed to whichever team's name triggered the Guardian search query. Match preview articles will mention both teams, so the home/away split is a heuristic. This limitation is addressed in `nlp_features.ipynb` via Named entity recognition.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH    = REPO_ROOT / 'data' / 'database.sqlite'
CACHE_PATH = REPO_ROOT / 'data' / 'articles_cache.json'

assert DB_PATH.exists(),    f'Missing {DB_PATH}'
assert CACHE_PATH.exists(), f'Missing {CACHE_PATH} — run articles.ipynb first'

print(f'Cache: {CACHE_PATH} ({CACHE_PATH.stat().st_size / 1024**2:.1f} MB)')

Cache: /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/articles_cache.json (74.4 MB)


## Define keyword lists

Three categories of signals, each matched as whole words (case-insensitive) to avoid false positives like "returning" matching "return".

In [3]:
KEYWORDS = {
    'injury': [
        'injured', 'injury', 'injuries', 'out', 'doubt', 'doubtful',
        'ruled out', 'fitness doubt', 'fitness concern', 'miss',
        'missing', 'unavailable', 'hamstring', 'knee', 'ankle',
        'muscle', 'strain', 'torn', 'fracture', 'concussion',
    ],
    'suspension': [
        'suspended', 'suspension', 'ban', 'banned', 'red card',
        'sent off', 'dismissal', 'accumulated', 'yellow card',
    ],
    'return': [
        'returns', 'returned', 'return', 'fit again', 'back in training',
        'back from injury', 'recovered', 'available again', 'recalled',
    ],
}

# Compile regex patterns for each category (whole word, case-insensitive)
PATTERNS = {
    category: re.compile(
        r'\b(' + '|'.join(re.escape(k) for k in keywords) + r')\b',
        re.IGNORECASE
    )
    for category, keywords in KEYWORDS.items()
}

print('Keyword patterns compiled:')
for cat, kws in KEYWORDS.items():
    print(f'  {cat}: {len(kws)} keywords')

Keyword patterns compiled:
  injury: 20 keywords
  suspension: 9 keywords
  return: 9 keywords


## Rebuild match-article records from cache

Load the cache and reconstruct the per-match article lists. This is fast since it only reads the local JSON file.

In [4]:
from datetime import timedelta
import unicodedata

LOOKBACK_DAYS = 7

with open(CACHE_PATH) as f:
    cache = json.load(f)
print(f'Cache loaded: {len(cache):,} entries')

conn = sqlite3.connect(DB_PATH)
matches = pd.read_sql(
    """
    SELECT m.match_api_id, m.date,
           ht.team_long_name AS home_team_name,
           at.team_long_name AS away_team_name
    FROM Match m
    JOIN Team ht ON ht.team_api_id = m.home_team_api_id
    JOIN Team at ON at.team_api_id = m.away_team_api_id
    ORDER BY m.date
    """,
    conn,
)
conn.close()
matches['date'] = pd.to_datetime(matches['date'])

def clean_team_name(name):
    return unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii').strip()

def get_cached_articles(team_name, match_date):
    from_date = (match_date - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    to_date   = (match_date - timedelta(days=1)).strftime('%Y-%m-%d')
    cache_key = f"{team_name}|{from_date}|{to_date}"
    return cache.get(cache_key, [])

print(f'Loaded {len(matches):,} matches')

Cache loaded: 48,128 entries
Loaded 25,979 matches


## Extract keyword features

For each match, count keyword hits across all article headlines and bodies for the home and away team separately.

In [5]:
def count_keywords(articles, pattern):
    """Count total keyword matches across all articles (headline + body)."""
    total = 0
    for article in articles:
        text = article.get('headline', '') + ' ' + article.get('body', '')
        total += len(pattern.findall(text))
    return total

records = []
for row in matches.itertuples():
    home_arts = get_cached_articles(row.home_team_name, row.date)
    away_arts = get_cached_articles(row.away_team_name, row.date)

    records.append({
        'match_api_id':          row.match_api_id,
        'home_injury_count':     count_keywords(home_arts, PATTERNS['injury']),
        'away_injury_count':     count_keywords(away_arts, PATTERNS['injury']),
        'home_suspension_count': count_keywords(home_arts, PATTERNS['suspension']),
        'away_suspension_count': count_keywords(away_arts, PATTERNS['suspension']),
        'home_return_count':     count_keywords(home_arts, PATTERNS['return']),
        'away_return_count':     count_keywords(away_arts, PATTERNS['return']),
        'home_has_coverage':     int(len(home_arts) > 0),
        'away_has_coverage':     int(len(away_arts) > 0),
    })

keyword_features = pd.DataFrame(records)
print(f'Keyword features shape: {keyword_features.shape}')
keyword_features.head(10)

Keyword features shape: (25979, 9)


,match_api_id,home_injury_count,away_injury_count,home_suspension_count,away_suspension_count,home_return_count,away_return_count,home_has_coverage,away_has_coverage
0,486263,0,18,0,4,0,3,0,1
1,486264,18,21,2,2,3,4,1,1
2,486265,14,14,2,2,0,0,1,1
3,486266,0,14,0,2,0,0,0,1
4,486267,7,11,1,0,0,1,1,1
5,486268,6,0,0,0,2,0,1,0
6,486269,7,7,1,1,0,0,1,1
7,486270,14,0,1,0,0,0,1,1
8,486271,14,14,1,1,0,0,1,1
9,486272,10,6,0,0,0,2,1,1


## Sanity check

Check the distribution of keyword counts and coverage rates across matches.

In [6]:
print('Coverage rates:')
print(f"  Home team has articles: {keyword_features['home_has_coverage'].mean():.1%}")
print(f"  Away team has articles: {keyword_features['away_has_coverage'].mean():.1%}")

print('\nKeyword count distributions (matches with coverage only):')
covered = keyword_features[keyword_features['home_has_coverage'] == 1]
cols = ['home_injury_count', 'away_injury_count',
        'home_suspension_count', 'away_suspension_count',
        'home_return_count', 'away_return_count']
print(covered[cols].describe().round(2))

Coverage rates:
  Home team has articles: 4.1%
  Away team has articles: 3.9%

Keyword count distributions (matches with coverage only):
       home_injury_count  away_injury_count  home_suspension_count  \
count            1053.00            1053.00                1053.00   
mean               26.74               4.43                   3.32   
std                25.44              13.49                   3.94   
min                 0.00               0.00                   0.00   
25%                 7.00               0.00                   0.00   
50%                21.00               0.00                   2.00   
75%                38.00               0.00                   5.00   
max               173.00             124.00                  28.00   

       away_suspension_count  home_return_count  away_return_count  
count                1053.00            1053.00            1053.00  
mean                    0.51               2.99               0.50  
std                     1

## Merge onto the modelling dataset

Merge keyword features onto the base dataset by `match_api_id`. Matches with no coverage get NaN keyword counts — filled with 0 since absence of articles means absence of signal, not missing data. The `has_coverage` flags are kept so the model can learn that uncovered matches are a distinct group.

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Rebuild base dataset (same pipeline as baseline.ipynb)
conn = sqlite3.connect(DB_PATH)

base = pd.read_sql(
    """
    SELECT match_api_id, date,
           home_team_api_id, away_team_api_id,
           home_team_goal, away_team_goal,
           home_player_1,  home_player_2,  home_player_3,
           home_player_4,  home_player_5,  home_player_6,
           home_player_7,  home_player_8,  home_player_9,
           home_player_10, home_player_11,
           away_player_1,  away_player_2,  away_player_3,
           away_player_4,  away_player_5,  away_player_6,
           away_player_7,  away_player_8,  away_player_9,
           away_player_10, away_player_11
    FROM Match ORDER BY date
    """, conn)

conditions = [
    base['home_team_goal'] > base['away_team_goal'],
    base['home_team_goal'] == base['away_team_goal'],
    base['home_team_goal'] < base['away_team_goal'],
]
base['result']      = np.select(conditions, ['Win', 'Draw', 'Defeat'], default='')
base['result_code'] = base['result'].map({'Defeat': 0, 'Draw': 1, 'Win': 2})
base['date']        = pd.to_datetime(base['date'])

# Player ratings
player_cols = [f'home_player_{i}' for i in range(1, 12)] + [f'away_player_{i}' for i in range(1, 12)]
match_players = (
    base[['match_api_id', 'date'] + player_cols]
    .melt(id_vars=['match_api_id', 'date'], value_vars=player_cols, value_name='player_api_id')
    .dropna(subset=['player_api_id']).astype({'player_api_id': int})
)
player_attrs = pd.read_sql('SELECT player_api_id, date, overall_rating FROM Player_Attributes', conn)
player_attrs['date']    = pd.to_datetime(player_attrs['date'])
match_players           = match_players.sort_values('date')
player_attrs            = player_attrs.sort_values('date')
rated = pd.merge_asof(match_players, player_attrs, on='date', by='player_api_id', direction='backward')
rated['side'] = rated['variable'].str.split('_player_').str[0]
mts = (rated.groupby(['match_api_id', 'side'])['overall_rating']
       .agg(avg_rating='mean', min_rating='min', max_rating='max', std_rating='std')
       .unstack('side'))
mts.columns = [f'{side}_{stat}' for stat, side in mts.columns]
mts = mts.reset_index()
base = base.merge(mts, on='match_api_id', how='left')
base = base.dropna(subset=['home_avg_rating', 'away_avg_rating'])

# Rolling form
from datetime import timedelta as td
def compute_team_form(df, n=5):
    home = df[['match_api_id','date','home_team_api_id','home_team_goal','away_team_goal']].copy()
    home.columns = ['match_api_id','date','team_id','goals_for','goals_against']
    home['is_home'] = True
    away = df[['match_api_id','date','away_team_api_id','away_team_goal','home_team_goal']].copy()
    away.columns = ['match_api_id','date','team_id','goals_for','goals_against']
    away['is_home'] = False
    tl = pd.concat([home, away], ignore_index=True).sort_values(['team_id','date'])
    tl['points'] = np.select([tl['goals_for']>tl['goals_against'], tl['goals_for']==tl['goals_against']], [3,1], default=0)
    for s in ['goals_for','goals_against','points']:
        tl[f'rolling_{s}'] = tl.groupby('team_id')[s].transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    hf = tl[tl['is_home']].rename(columns={'rolling_goals_for':'home_rolling_gf','rolling_goals_against':'home_rolling_ga','rolling_points':'home_rolling_pts'})[['match_api_id','home_rolling_gf','home_rolling_ga','home_rolling_pts']]
    af = tl[~tl['is_home']].rename(columns={'rolling_goals_for':'away_rolling_gf','rolling_goals_against':'away_rolling_ga','rolling_points':'away_rolling_pts'})[['match_api_id','away_rolling_gf','away_rolling_ga','away_rolling_pts']]
    return hf.merge(af, on='match_api_id')
team_form = compute_team_form(base)

# H2H
def compute_h2h(df, n=5):
    df_sorted = df.sort_values('date').reset_index(drop=True)
    df_sorted['_t1'] = np.minimum(df_sorted['home_team_api_id'], df_sorted['away_team_api_id'])
    df_sorted['_t2'] = np.maximum(df_sorted['home_team_api_id'], df_sorted['away_team_api_id'])
    records = []
    for idx, row in df_sorted.iterrows():
        prior = df_sorted.iloc[:idx]
        prior = prior[(prior['_t1']==row['_t1'])&(prior['_t2']==row['_t2'])].tail(n)
        if prior.empty:
            records.append({'match_api_id':row['match_api_id'],'h2h_home_win_rate':np.nan,'h2h_avg_goal_diff':np.nan,'h2h_n_matches':0})
            continue
        h = row['home_team_api_id']
        diff = np.where(prior['home_team_api_id']==h, prior['home_team_goal']-prior['away_team_goal'], prior['away_team_goal']-prior['home_team_goal'])
        records.append({'match_api_id':row['match_api_id'],'h2h_home_win_rate':(diff>0).sum()/len(prior),'h2h_avg_goal_diff':diff.mean(),'h2h_n_matches':len(prior)})
    return pd.DataFrame(records)
print('Computing H2H (may take ~30s)...')
h2h = compute_h2h(base)

# Team attributes
TACTICAL_COLS = ['buildUpPlaySpeed','buildUpPlayPassing','chanceCreationPassing','chanceCreationCrossing','chanceCreationShooting','defencePressure','defenceAggression','defenceTeamWidth']
team_attrs = pd.read_sql(f"SELECT team_api_id, date, {', '.join(TACTICAL_COLS)} FROM Team_Attributes", conn)
conn.close()
team_attrs['date'] = pd.to_datetime(team_attrs['date'])
team_attrs = team_attrs.sort_values('date')
base_sorted = base.sort_values('date')
def merge_attrs(df, team_col, prefix):
    m = pd.merge_asof(df[['match_api_id','date',team_col]], team_attrs, on='date', left_by=team_col, right_by='team_api_id', direction='backward').drop(columns=['team_api_id',team_col])
    return m.rename(columns={c:f'{prefix}_{c}' for c in TACTICAL_COLS})
home_attrs = merge_attrs(base_sorted, 'home_team_api_id', 'home')
away_attrs = merge_attrs(base_sorted, 'away_team_api_id', 'away')

# Assemble base dataset
RATING_COLS = ['home_avg_rating','away_avg_rating','home_min_rating','away_min_rating','home_max_rating','away_max_rating','home_std_rating','away_std_rating']
home_tac = [f'home_{c}' for c in TACTICAL_COLS]
away_tac = [f'away_{c}' for c in TACTICAL_COLS]
dataset = (
    base[['match_api_id','date','result','result_code']+RATING_COLS]
    .merge(team_form, on='match_api_id', how='left')
    .merge(h2h,       on='match_api_id', how='left')
    .merge(home_attrs[['match_api_id']+home_tac], on='match_api_id', how='left')
    .merge(away_attrs[['match_api_id']+away_tac], on='match_api_id', how='left')
)

# Merge keyword features
KEYWORD_COLS = ['home_injury_count','away_injury_count','home_suspension_count','away_suspension_count','home_return_count','away_return_count','home_has_coverage','away_has_coverage']
dataset = dataset.merge(keyword_features[['match_api_id']+KEYWORD_COLS], on='match_api_id', how='left')

# Fill NaN keyword counts with 0 (no coverage = no signal)
dataset[KEYWORD_COLS] = dataset[KEYWORD_COLS].fillna(0)

print(f'Final dataset shape: {dataset.shape}')
print(f'\nKeyword feature null rates:')
print(dataset[KEYWORD_COLS].isnull().mean().round(3))
dataset.head()

Computing H2H (may take ~30s)...
Final dataset shape: (25221, 45)

Keyword feature null rates:
home_injury_count        0.0
away_injury_count        0.0
home_suspension_count    0.0
away_suspension_count    0.0
home_return_count        0.0
away_return_count        0.0
home_has_coverage        0.0
away_has_coverage        0.0
dtype: float64


,match_api_id,date,result,result_code,home_avg_rating,away_avg_rating,home_min_rating,away_min_rating,home_max_rating,away_max_rating,...,away_defenceAggression,away_defenceTeamWidth,home_injury_count,away_injury_count,home_suspension_count,away_suspension_count,home_return_count,away_return_count,home_has_coverage,away_has_coverage
0,483129,2008-08-09,Win,2,70.000000,69.200000,51.0,61.0,81.0,81.0,...,NaN,NaN,0,32,0,3,0,2,0,1
1,483130,2008-08-09,Win,2,74.454545,69.454545,65.0,58.0,83.0,77.0,...,NaN,NaN,4,4,1,1,1,1,1,1
2,483131,2008-08-09,Win,2,64.909091,68.818182,47.0,55.0,77.0,75.0,...,NaN,NaN,24,0,1,0,7,0,1,0
3,483132,2008-08-09,Defeat,0,70.444444,67.181818,61.0,58.0,76.0,75.0,...,NaN,NaN,26,33,1,2,4,1,1,1
4,483134,2008-08-09,Win,2,67.909091,73.363636,52.0,51.0,76.0,85.0,...,NaN,NaN,4,11,1,1,1,2,1,1


## Retrain baseline models with keyword features

Temporal split — pre-2014 train, 2014 onwards test. Compare accuracy against the baseline results from `baseline.ipynb` to measure the keyword features' contribution.

In [8]:
FEATURE_COLS = (
    RATING_COLS
    + ['home_rolling_gf','home_rolling_ga','home_rolling_pts',
       'away_rolling_gf','away_rolling_ga','away_rolling_pts']
    + ['h2h_home_win_rate','h2h_avg_goal_diff']
    + home_tac + away_tac
    + KEYWORD_COLS
)

train = dataset[dataset['date'] < '2014-01-01'].copy()
test  = dataset[dataset['date'] >= '2014-01-01'].copy()

train_medians = train[FEATURE_COLS].median()
X_train = train[FEATURE_COLS].fillna(train_medians)
X_test  = test[FEATURE_COLS].fillna(train_medians)
y_train = train['result_code']
y_test  = test['result_code']

label_names = ['Defeat', 'Draw', 'Win']
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

Train: 17,051  |  Test: 8,170


In [9]:
from sklearn.metrics import f1_score

gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42)
rfc = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=11, metric='euclidean', n_jobs=-1)

print('Training GBC...'); gbc.fit(X_train, y_train)
print('Training RFC...'); rfc.fit(X_train, y_train)
print('Training KNN...'); knn.fit(X_train_scaled, y_train)
print('Done.')

Training GBC...
Training RFC...
Training KNN...
Done.


In [10]:
results = pd.DataFrame([
    {
        'Model': 'Gradient Boosting',
        'Accuracy': accuracy_score(y_test, gbc.predict(X_test)),
        'F1 (macro)': f1_score(y_test, gbc.predict(X_test), average='macro'),
        'F1 (weighted)': f1_score(y_test, gbc.predict(X_test), average='weighted'),
    },
    {
        'Model': 'Random Forest',
        'Accuracy': accuracy_score(y_test, rfc.predict(X_test)),
        'F1 (macro)': f1_score(y_test, rfc.predict(X_test), average='macro'),
        'F1 (weighted)': f1_score(y_test, rfc.predict(X_test), average='weighted'),
    },
    {
        'Model': 'KNN (k=11)',
        'Accuracy': accuracy_score(y_test, knn.predict(X_test_scaled)),
        'F1 (macro)': f1_score(y_test, knn.predict(X_test_scaled), average='macro'),
        'F1 (weighted)': f1_score(y_test, knn.predict(X_test_scaled), average='weighted'),
    },
]).set_index('Model').round(3)

print('Results WITH keyword features:')
print(results)
print('\nBaseline (no keyword features) from baseline.ipynb:')
print('  GBC: 0.512 accuracy')

Results WITH keyword features:
                   Accuracy  F1 (macro)  F1 (weighted)
Model                                                 
Gradient Boosting     0.512       0.381          0.437
Random Forest         0.512       0.372          0.430
KNN (k=11)            0.445       0.393          0.428

Baseline (no keyword features) from baseline.ipynb:
  GBC: 0.512 accuracy


Here, we see the accuracy is relatively unchanged. This is because a raw summation is unable to account for the duplicate injury mentions and cannot correlate which player belongs to each team. Often sports articles contain match information, so mention both the home and away team, so a count is a bad heuristic.